# Ejemplo desarrollado — SQL + pandas
### Unidad 4: SQL para Ciencia de Datos

En este notebook vamos a resolver **de forma completa** un análisis de ventas combinando
**SQL** (usando `sqlite3` y `pandas.read_sql_query`) con **pandas puro**, tal como vimos en
la unidad: `SELECT`, `WHERE`, `GROUP BY`, `HAVING`, `ORDER BY`.

**Objetivo:** a partir de una tabla de clientes con sus ventas, vamos a:
1. Crear la base de datos y cargar los datos.
2. Consultar y filtrar datos con `WHERE`.
3. Agrupar y agregar datos con `GROUP BY` + funciones de agregación.
4. Resolver las mismas consultas con pandas, sin SQL, y comparar resultados.


## 1. Preparar el entorno y los datos

In [1]:
import sqlite3
import pandas as pd

# Creamos una base de datos SQLite en memoria (no genera archivo en disco)
conn = sqlite3.connect(':memory:')

# Dataset de clientes con sus ventas (simula la tabla "clientes" de la Unidad 4)
clientes = pd.DataFrame({
    'id':      [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'nombre':  ['Ana López', 'Bruno Díaz', 'Carla Ruiz', 'Diego Paz', 'Elena Vidal',
                'Franco Sosa', 'Gabriela Mora', 'Hugo Vega', 'Irene Castro', 'Juan Pérez'],
    'ciudad':  ['Buenos Aires', 'Córdoba', 'Buenos Aires', 'Rosario', 'Córdoba',
                'Buenos Aires', 'Rosario', 'Córdoba', 'Buenos Aires', 'Rosario'],
    'ventas':  [15000, 8700, 21000, 5400, 12300, 30500, 0, 9800, 17600, 4200],
    'email':   ['ana@mail.com', 'bruno@mail.com', 'carla@mail.com', 'diego@gmail.com',
                'elena@mail.com', 'franco@mail.com', 'gabriela@gmail.com', 'hugo@mail.com',
                'irene@mail.com', 'juan@gmail.com']
})

# DDL + carga: creamos la tabla en SQLite a partir del DataFrame
clientes.to_sql('clientes', conn, index=False, if_exists='replace')

clientes

,id,nombre,ciudad,ventas,email
0,1,Ana López,Buenos Aires,15000,ana@mail.com
1,2,Bruno Díaz,Córdoba,8700,bruno@mail.com
2,3,Carla Ruiz,Buenos Aires,21000,carla@mail.com
3,4,Diego Paz,Rosario,5400,diego@gmail.com
4,5,Elena Vidal,Córdoba,12300,elena@mail.com
5,6,Franco Sosa,Buenos Aires,30500,franco@mail.com
6,7,Gabriela Mora,Rosario,0,gabriela@gmail.com
7,8,Hugo Vega,Córdoba,9800,hugo@mail.com
8,9,Irene Castro,Buenos Aires,17600,irene@mail.com
9,10,Juan Pérez,Rosario,4200,juan@gmail.com


## 2. `SELECT` + `WHERE` — filtrar filas

Queremos los clientes con ventas mayores a $10.000 y cuyo email sea de dominio `@mail.com`
(igual que el ejemplo de la diapositiva 9, usando `BETWEEN`, `LIKE` y `AND`).

In [2]:
query_where = '''
SELECT nombre, ciudad, ventas
FROM clientes
WHERE ventas > 10000
  AND email LIKE '%@mail.com'
ORDER BY ventas DESC;
'''

df_where = pd.read_sql_query(query_where, conn)
df_where

,nombre,ciudad,ventas
0,Franco Sosa,Buenos Aires,30500
1,Carla Ruiz,Buenos Aires,21000
2,Irene Castro,Buenos Aires,17600
3,Ana López,Buenos Aires,15000
4,Elena Vidal,Córdoba,12300


**Equivalente 100% en pandas** (sin SQL), para comparar ambos enfoques:

In [3]:
df_where_pandas = (
    clientes[(clientes['ventas'] > 10000) & (clientes['email'].str.endswith('@mail.com'))]
    [['nombre', 'ciudad', 'ventas']]
    .sort_values('ventas', ascending=False)
    .reset_index(drop=True)
)
df_where_pandas

,nombre,ciudad,ventas
0,Franco Sosa,Buenos Aires,30500
1,Carla Ruiz,Buenos Aires,21000
2,Irene Castro,Buenos Aires,17600
3,Ana López,Buenos Aires,15000
4,Elena Vidal,Córdoba,12300


In [4]:
# Verificamos que ambos resultados sean equivalentes
assert df_where.reset_index(drop=True).equals(df_where_pandas), "Los resultados no coinciden"
print("✅ SQL y pandas dan el mismo resultado")

✅ SQL y pandas dan el mismo resultado


## 3. `GROUP BY` + funciones de agregación

Replicamos el ejemplo de la diapositiva 10: total de ventas y cantidad de clientes por
ciudad, quedándonos solo con ciudades que tengan **3 o más clientes** (`HAVING`),
ordenado de mayor a menor total.

In [5]:
query_group = '''
SELECT
    ciudad,
    SUM(ventas)   AS total,
    COUNT(*)      AS clientes,
    ROUND(AVG(ventas), 2) AS promedio
FROM clientes
GROUP BY ciudad
HAVING COUNT(*) >= 3
ORDER BY total DESC;
'''

df_group = pd.read_sql_query(query_group, conn)
df_group

,ciudad,total,clientes,promedio
0,Buenos Aires,84100,4,21025.00
1,Córdoba,30800,3,10266.67
2,Rosario,9600,3,3200.00


**Equivalente en pandas**, usando `groupby().agg()` y `query()` para el `HAVING`:

In [6]:
df_group_pandas = (
    clientes.groupby('ciudad')
    .agg(total=('ventas', 'sum'),
         clientes=('ventas', 'count'),
         promedio=('ventas', 'mean'))
    .round(2)
    .query('clientes >= 3')          # equivalente al HAVING
    .sort_values('total', ascending=False)
    .reset_index()
)
df_group_pandas

,ciudad,total,clientes,promedio
0,Buenos Aires,84100,4,21025.00
1,Córdoba,30800,3,10266.67
2,Rosario,9600,3,3200.00


## 4. Conclusión

- `WHERE` en SQL  ➜  filtrado booleano (`df[condición]`) en pandas.
- `GROUP BY` + agregaciones  ➜  `groupby().agg()` en pandas.
- `HAVING`  ➜  filtrar el resultado del `groupby` con `.query()` o indexado booleano.
- `ORDER BY`  ➜  `.sort_values()`.

Ambos caminos llegan al mismo resultado: SQL es útil cuando los datos viven en una base
de datos relacional y querés aprovechar su motor de consultas; pandas es útil cuando ya
tenés los datos en memoria como DataFrame y querés seguir encadenando transformaciones
en Python.

In [7]:
conn.close()